In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "5"
import textattack
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd
from langdetect import detect

c:\Users\Shaz\interp-toxicity\interp-toxicity\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [27]:
df = pd.read_csv('all_data.csv')

In [28]:
df = df[df.male.notna()].reset_index(drop=True)

In [29]:
df.to_csv('jigsaw/test_with_dem.csv', index=False)

In [7]:
df2 = pd.read_json('snlp_roberta/jigsaw_dem_pgd.jsonl', lines=True)

In [25]:
df = df[df.comment_text.isin(df2.orig_text)]

In [26]:
df2.drop_duplicates('orig_text')

,id,orig_text,adv_text,orig_label,success,iters,error,is_intermediate
0,0,"Thank you Stephanie, great post. You should be...","[UNK' you Stephanie, great post. You should be...",0,False,10,NaN,True
1,1,"Poor, poor us! We're so abused! Give it a brea...","dat, poor us! We're so abused! Give it a break...",0,True,1,NaN,True
2,2,"Yeah, as we've gotten so used to with the Rock...","Yeah, as we've gotten so used to with the Rock...",0,True,5,NaN,True
3,3,"If your actions thereby assist Sears' failure,...","Ifthethou,is Sears' failure, you will contribu...",0,False,10,NaN,True
4,4,"It is indeed a balancing act, but by supportin...","It is indeed a balancing act, but by supportin...",0,False,10,NaN,True
...,...,...,...,...,...,...,...,...
16796,16796,"Asphyxiation is out (during courting, anyway),...","Asex; is out (during courting, anyway), check....",0,False,10,NaN,True
16797,16797,Only until the Pope starts claiming spiritual ...,Only until the Pope starts claiming spiritual ...,0,False,10,NaN,True
16798,16798,Like all Trump Bunnies you are ignorant. Trum...,Like all Trump Bunnies you areg. Trump isn't ...,1,True,1,NaN,True
16799,16799,Tired right wing talking point.\n\nWe have law...,Tired right wing talking point.\n\nWe have law...,0,False,10,NaN,True


In [18]:
# Find duplicates in df
duplicates = df.duplicated(subset='comment_text', keep=False)
print(f"Number of duplicate rows: {duplicates.sum()}")
print(f"Number of unique comment_text values: {df['comment_text'].nunique()}")
print(f"Total rows: {len(df)}")

# Show duplicate comment_text entries
if duplicates.any():
    duplicate_texts = df[duplicates].groupby('comment_text').size().sort_values(ascending=False)
    print(f"\nTop duplicate comment_text entries:")
    print(duplicate_texts.head(10))


Number of duplicate rows: 599
Number of unique comment_text values: 16790
Total rows: 17227

Top duplicate comment_text entries:
comment_text
No.                                                                                                                                                                                                                                                                                                                                                          27
Yes.                                                                                                                                                                                                                                                                                                                                                         20
What?                                                                                                                                                     

In [17]:
df.drop_duplicates('comment_text')

,id,comment_text,split,created_date,publication_id,parent_id,article_id,rating,funny,wow,...,white,asian,latino,other_race_or_ethnicity,physical_disability,intellectual_or_learning_disability,psychiatric_or_mental_illness,other_disability,identity_annotator_count,toxicity_annotator_count
30,6084809,What a trite ignorant letter that repeats the ...,train,2017-10-04 22:59:43.824399+00,13,NaN,385392,approved,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,74
31,6121135,How do we fight agaisnt women who use sexual f...,train,2017-10-10 21:45:32.903437+00,54,NaN,386192,rejected,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,10
53,525440,Yes but it was her power and right as an emplo...,train,2016-10-13 07:22:29.620512+00,21,525379.0,148341,approved,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,10
81,454741,Sucker born every minute.,train,2016-08-31 14:08:42.223142+00,13,NaN,144907,approved,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,59
85,642230,you are going to be sent to a home for the stu...,train,2016-11-30 23:46:40.996032+00,54,641911.0,153630,rejected,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,51
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
447879,935715,If all the left has to offer are homophobic an...,train,2017-02-01 23:05:53.405456+00,54,NaN,165522,rejected,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10,10
447890,5907486,"We don't call ""anyone"" racist Lars...just peop...",train,2017-09-06 15:52:47.230624+00,21,5906716.0,374500,approved,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10,10
447913,5915140,So we can assume that Austin Ruse has some ser...,train,2017-09-07 17:36:30.932375+00,53,NaN,375168,approved,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10,10
447923,622896,At this account Jesus is extending mercy to a ...,train,2016-11-23 19:16:35.751879+00,53,622858.0,151612,approved,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10,10


In [14]:
from tqdm import tqdm, trange

from datasets import load_dataset
import pandas as pd
import functools
import sys
from pathlib import Path
from typing import Callable

# import circuitsvis as cv
import einops
import numpy as np
import torch as t
import torch.nn as nn
import torch.nn.functional as F
import eindex
# from IPython.display import display
from jaxtyping import Float, Int
from torch import Tensor
from tqdm import tqdm
# from transformer_lens import (
#     ActivationCache,
#     FactoredMatrix,
#     HookedTransformer,
#     HookedTransformerConfig,
#     HookedEncoderDecoder,
#     HookedEncoder,
#     utils,
# )
# from transformer_lens.hook_points import HookPoint

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer
# from transformer_lens import HookedTransformer
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
tqdm.pandas()

In [26]:
print("Available CUDA devices:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}: {torch.cuda.get_device_name(i)}")
device = torch.device('cuda:3') if torch.cuda.is_available() else torch.device('cpu')

Available CUDA devices: 8
Device 0: NVIDIA H100 80GB HBM3
Device 1: NVIDIA H100 80GB HBM3
Device 2: NVIDIA H100 80GB HBM3
Device 3: NVIDIA H100 80GB HBM3
Device 4: NVIDIA H100 80GB HBM3
Device 5: NVIDIA H100 80GB HBM3
Device 6: NVIDIA H100 80GB HBM3
Device 7: NVIDIA H100 80GB HBM3


In [27]:
tokenizer = AutoTokenizer.from_pretrained("s-nlp/roberta_toxicity_classifier")
model = AutoModelForSequenceClassification.from_pretrained("s-nlp/roberta_toxicity_classifier")

Some weights of the model checkpoint at s-nlp/roberta_toxicity_classifier were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## Baseline

In [28]:
df = pd.read_csv('all_cases.csv')

In [32]:
df['label'] = df.label_gold.apply(lambda x: 1 if x == 'hateful' else 0)

In [35]:
from sklearn.metrics import accuracy_score
from tqdm import trange

# Batch size for GPU inference
BATCH_SIZE = 64

all_preds = []
all_labels = []

model = model.to(device)
model.eval()
new_df = df
num_samples = len(new_df)
for start_idx in trange(0, num_samples, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, num_samples)
    batch_texts = new_df.test_case.iloc[start_idx:end_idx].tolist()
    batch_labels = new_df.label.iloc[start_idx:end_idx].astype(int).tolist()
    inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).cpu().tolist()
    all_preds.extend(preds)
    all_labels.extend(batch_labels)

accuracy = accuracy_score(all_labels, all_preds)
print(f"Baseline model accuracy on test set: {accuracy:.4f}")

100%|██████████| 61/61 [00:01<00:00, 51.79it/s]

Baseline model accuracy on test set: 0.7214


## Attack

In [11]:
# Import our PGD attack implementation
from pgd_bert_attack import PGDBERTAttack, set_seed

# Set seed for reproducibility
set_seed(42)

In [13]:
# Initialize PGD Attack
print("Setting up PGD BERT Attack...")

# Create the attacker - it will automatically load BERT MLM model for candidate generation
attacker = PGDBERTAttack(
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_iters=10,
    top_k_tokens=5,
    mlm_top_k=50,
    sim_threshold=0.9,  # Slightly lower threshold for more flexibility
    max_length=512
)

print("PGD Attack setup complete!")


Setting up PGD BERT Attack...
Loading BERT MLM model for candidate generation...


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


PGD Attack setup complete!


In [16]:
# Evaluate model robustness on a sample of data
print("Evaluating model robustness with PGD attacks...")

# Sample some data for evaluation
sample_size = 100  # Reduced for demonstration
sample_df = df.sample(sample_size, random_state=42)
sample_texts = sample_df.comment_text.tolist()
sample_labels = sample_df.toxic.astype(int).tolist()

print(f"Running attacks on {sample_size} samples...")
print(f"Sample distribution: {sum(sample_labels)} toxic, {len(sample_labels) - sum(sample_labels)} non-toxic")

# Run batch attack evaluation with saving enabled
output_file = "snlp_roberta/jigsaw_pgd.jsonl"
robustness_results = attacker.evaluate_robustness(
    sample_texts, 
    sample_labels,
    output_file=output_file,
    log_interval=100  # Save every 25 examples
)

print(f"\nRobustness Evaluation Results:")
print(f"Total samples: {robustness_results['total_samples']}")
print(f"Successful attacks: {robustness_results['successful_attacks']}")
print(f"Attack success rate: {robustness_results['attack_success_rate']:.2%}")
print(f"Average iterations for successful attacks: {robustness_results['average_iterations']:.1f}")
print(f"Results saved to: {output_file}")


Evaluating model robustness with PGD attacks...
Running attacks on 100 samples...
Sample distribution: 10 toxic, 90 non-toxic
Evaluating robustness on 100 samples...


Attacking:   0%|          | 0/100 [00:00<?, ?it/s]

Loading sentence transformer for similarity checking...


Attacking: 100%|██████████| 100/100 [10:33<00:00,  6.34s/it]

Processed 100/100, Success rate: 37.00%
Saved intermediate results (100 examples) → snlp_roberta/jigsaw_pgd.jsonl
Saved final results (100 examples) → snlp_roberta/jigsaw_pgd.jsonl
Final attack success rate: 37.00%

Robustness Evaluation Results:
Total samples: 100
Successful attacks: 37
Attack success rate: 37.00%
Average iterations for successful attacks: 3.5
Results saved to: snlp_roberta/jigsaw_pgd.jsonl


In [17]:
# Analyze attack results and show examples
print("Analysis of Attack Results:")
print("=" * 50)

results = robustness_results['results']
successful_attacks = [(i, adv_text, meta) for i, (adv_text, meta) in enumerate(results) if meta['success']]

if successful_attacks:
    print(f"\nShowing first 5 successful attacks:")
    print("-" * 40)
    
    for i, (idx, adv_text, meta) in enumerate(successful_attacks[:5]):
        original_text = meta['original_text']
        print(f"\nExample {i+1}:")
        print(f"Original: {original_text}")
        print(f"Adversarial: {adv_text}")
        print(f"Original label: {sample_labels[idx]}")
        print(f"Iterations: {meta['iters']}")
        
        # Check the difference
        from difflib import SequenceMatcher
        similarity = SequenceMatcher(None, original_text, adv_text).ratio()
        print(f"Text similarity: {similarity:.3f}")
        print("-" * 40)
else:
    print("No successful attacks found in this sample.")

# Attack success by label
print(f"\nAttack Success by Original Label:")
toxic_attacks = [(adv_text, meta) for i, (adv_text, meta) in enumerate(results) if sample_labels[i] == 1]
non_toxic_attacks = [(adv_text, meta) for i, (adv_text, meta) in enumerate(results) if sample_labels[i] == 0]

if toxic_attacks:
    toxic_success_rate = sum(1 for _, meta in toxic_attacks if meta['success']) / len(toxic_attacks)
    print(f"Toxic examples: {toxic_success_rate:.2%} success rate ({len(toxic_attacks)} samples)")

if non_toxic_attacks:
    non_toxic_success_rate = sum(1 for _, meta in non_toxic_attacks if meta['success']) / len(non_toxic_attacks)
    print(f"Non-toxic examples: {non_toxic_success_rate:.2%} success rate ({len(non_toxic_attacks)} samples)")

print(f"\nOverall model robustness: {100 - robustness_results['attack_success_rate']*100:.1f}% robust to PGD attacks")


Analysis of Attack Results:

Showing first 5 successful attacks:
----------------------------------------

Example 1:
Original: Barnstar for you  

  stylebackgroundcolor fdffe7 border 1px solid fceb92 
 rowspan2 styleverticalalign middle padding 5px   
 stylefontsize xlarge padding 3px 3px 0 3px height 15em  The Persian Merit Barnstar 
  
 styleverticalalign middle padding 3px  For your contributions to Persian language topics
Adversarial: Barnstar for you  

  stylebackgroundcolor fdffe7 border 1px solid fceb92 
 rowspan2 styleverticalalign middle padding 5px   
 stylefontsize xlarge padding 3px 3px 0 3px height 15em  The Persian Merit Barnstar 
  
 styleverticalalign middle padding 3px  For youuck,ag!
Original label: 0
Iterations: 7
Text similarity: 0.914
----------------------------------------

Example 2:
Original: lack of Mark Lavin  New York Post contrversy in the artical  

 Its strange there is nothing in the artical about the Mark Lavin exposing oberman about his alegedly bei

In [ ]:
# Demonstrate loading and analyzing saved results
print("Loading and analyzing saved attack results...")

# You can load results from a previous run
if output_file:
    # Analyze the saved results
    analysis = attacker.analyze_saved_results(output_file)
    
    print(f"\nDetailed breakdown:")
    print(f"- Toxic examples attacked: {analysis.get('toxic_samples', 0)}")
    print(f"- Non-toxic examples attacked: {analysis.get('non_toxic_samples', 0)}")
    print(f"- Robustness score: {100 - analysis.get('attack_success_rate', 0)*100:.1f}%")
    
    # You can also manually load the raw data for custom analysis
    raw_results = attacker.load_saved_results(output_file)
    print(f"\nFirst saved result example:")
    if raw_results:
        first_result = raw_results[0]
        print(f"ID: {first_result.get('id')}")
        print(f"Original: {first_result.get('orig_text', '')[:100]}...")
        print(f"Adversarial: {first_result.get('adv_text', '')[:100]}...")
        print(f"Success: {first_result.get('success')}")
        print(f"Iterations: {first_result.get('iters')}")
else:
    print("No output file specified - skipping analysis demo")


In [1]:
import pandas as pd
import json

with open("snlp_roberta/jigsaw_pgd.jsonl", "r") as f:
    jigsaw_pgd = pd.DataFrame([json.loads(line) for line in f])

In [36]:
jigsaw_pgd.success.mean()

0.6972972972972973

In [2]:
len(jigsaw_pgd)

8400

In [38]:
jigsaw_pgd

,success,iters,id,orig_text,adv_text,orig_label,error
0,False,10.0,0,Thank you for understanding I think very highl...,Thank you for Alt I think veryattled of you an...,0,NaN
1,True,2.0,1,Dear god this site is horrible,contr HELL this site is horrible,0,NaN
2,True,1.0,2,Somebody will invariably try to add Religion ...,Somebody will invariably try to add Religion ...,0,NaN
3,False,10.0,3,It says it right there that it IS a type The T...,It says it right there that it IS a type The T...,0,NaN
4,True,5.0,4,Before adding a new product to the list make s...,Before adding a new product to the list make s...,0,NaN
...,...,...,...,...,...,...,...
7395,True,10.0,7395,And this meets a much higher standard than ite...,And this meets a much higherallsALT Fat HELL H...,0,NaN
7396,True,1.0,7396,That is how Wikipedia works though I very muc...,That is how Wikipedia works though I very muc...,0,NaN
7397,False,10.0,7397,AF astronauts \n I decided to work this next C...,OP astronautsucker\n I decided to HELL this do...,0,NaN
7398,False,10.0,7398,I have changed the spelling I think this was d...,I have hung HELL Himself I think this was due ...,0,NaN


In [25]:
pd.read_csv('toxigen/test_clean.csv').generation[0]

'no matter how much money or power you have, if youre not white, you will never be considered a true american'